##**Final Project - FitCoach - фітнес-тренер**
Курс: AI Fundamentals · Product Management

1.Запускай комірки по порядку зверху вниз (Shift+Enter)



##0. Налаштування

In [ ]:
# КРОК 1. Підготовка середовища та встановлення бібліотек
!pip install -q \
    "langchain==1.3.11" \
    "langchain-openai==1.3.3" \
    "langchain-community==0.4.2" \
    "langgraph==1.2.7" \
    "langchainhub==0.1.21" \
    "python-dotenv==1.2.2" \
    "wikipedia==1.4.0" \
    "numexpr==2.14.1" \
    "numpy<3" 2>&1 | grep -v "dependency conflicts"

print(" Бібліотеки успішно встановлено.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.9/246.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.1 MB/s eta 0:00:00
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
 Бібліотеки успішно встановлено.


In [ ]:
# КРОК 2. Імпорт необхідних модулів
import os
import re
from datetime import datetime
from typing import List, Dict, Any
from dotenv import load_dotenv

# Модель OpenAI та інструменти LangChain
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool

print(" Імпорти успішно виконано.")

 Імпорти успішно виконано.


In [ ]:
# КРОК 3. Налаштування API-ключа OpenAI через Colab Secrets
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print(" Ключ завантажено з Colab Secrets")
except Exception:
    import getpass
    api_key = getpass.getpass("Введіть OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = api_key
    print(" Ключ встановлено")

 Ключ завантажено з Colab Secrets


##Допоміжні функції та бази даних

In [ ]:
# ==========================================
# ДОПОМІЖНІ ФУНКЦІЇ ТА ЛОКАЛЬНІ БАЗИ ДАНИХ
# ==========================================

def calculate_bmr(weight_kg: float, height_cm: float, age: int, gender: str) -> float:
    """Формула Міффліна-Сан Жеора для розрахунку базового метаболізму."""
    if gender.lower() in ["ч", "чоловік", "m", "male", "чол"]:
        return 10 * weight_kg + 6.25 * height_cm - 5 * age + 5
    else:
        return 10 * weight_kg + 6.25 * height_cm - 5 * age - 161

ACTIVITY_LEVELS = {
    "мінімальна": 1.2,
    "легка": 1.375,
    "помірна": 1.55,
    "висока": 1.725,
    "екстремальна": 1.9,
}

# База вправ — 6 груп м'язів, наявні вправи для дому та залу!
EXERCISES_DB = {
    "груди": {
        "вдома": [
            {"назва": "Віджимання від підлоги", "складність": "початківець", "техніка": "Руки на ширині плечей, тіло пряме", "підходи": "3-4", "повторення": "10-15"},
            {"назва": "Віджимання широким хватом", "складність": "середній", "техніка": "Руки ширше плечей", "підходи": "3", "повторення": "8-12"},
        ],
        "зал": [
            {"назва": "Жим штанги лежачи", "складність": "середній", "техніка": "Лопатки зведені, стопи на підлозі", "підходи": "4", "повторення": "8-12"},
        ]
    },
    "спина": {
        "вдома": [
            {"назва": "Супермен", "складність": "початківець", "техніка": "Лежачи на животі, піднімайте прямі руки та ноги", "підходи": "3", "повторення": "15-20"},
            {"назва": "Обернені віджимання від стільця", "складність": "початківець", "техніка": "Руки на стільці ззаду", "підходи": "3", "повторення": "10-12"},
        ]
    },
    "ноги": {
        "вдома": [
            {"назва": "Присідання", "складність": "початківець", "техніка": "Спина пряма, присідайте до паралелі", "підходи": "4", "повторення": "15-20"},
            {"назва": "Випади", "складність": "початківець", "техніка": "Крок вперед, заднє коліно майже торкається підлоги", "підходи": "3", "повторення": "12 на ногу"},
        ],
        "зал": [
            {"назва": "Жим ногами в тренажері", "складність": "середній", "техніка": "Спина притиснута до спинки, коліна під кутом 90 градусів", "підходи": "4", "повторення": "10-12"},
            {"назва": "Згинання ніг у тренажері лежачи", "складність": "початківець", "техніка": "Повільно згинайте ноги, фокус на задній поверхні стегна", "підходи": "3", "повторення": "12-15"},
        ]
    },
    "прес": {
        "вдома": [
            {"назва": "Скручування", "складність": "початківець", "техніка": "Лежачи на спині, піднімайте лише лопатки", "підходи": "3", "повторення": "20"},
            {"назва": "Планка", "складність": "початківець", "техніка": "Упор на лікті, тіло в лінію", "підходи": "3", "повторення": "30-60 сек"},
        ]
    },
    "руки": {
        "вдома": [
            {"назва": "Віджимання на трицепс", "складність": "початківець", "техніка": "Упор ззаду на стілець, згинайте лікті", "підходи": "3", "повторення": "10-15"},
            {"назва": "Віджимання вузьким хватом", "складність": "середній", "техніка": "Руки вузько, лікті вздовж тулуба", "підходи": "3", "повторення": "8-12"},
        ],
        "зал": [
            {"назва": "Підйом штанги на біцепс", "складність": "початківець", "техніка": "Спина пряма, лікті притиснуті до тулуба", "підходи": "3", "повторення": "10-12"},
            {"назва": "Розгинання рук на блоці (трицепс)", "складність": "початківець", "техніка": "Лікті зафіксовані, рух тільки в передпліччі", "підходи": "3", "повторення": "12-15"},
        ]
    },
    "плечі": {
        "вдома": [
            {"назва": "Віджимання 'будиночком' (Pike Push-ups)", "складність": "середній", "техніка": "Позиція планки, таз вгору, згинайте лікті", "підходи": "3", "повторення": "8-12"},
            {"назва": "Махи з пляшками", "складність": "початківець", "техніка": "Піднімайте руки через сторони до рівня плечей", "підходи": "3-4", "повторення": "15-20"},
        ]
    }
}

# Підрахунок загальної кількості вправ для гарного виводу
total_exercises = sum(len(ex) for locs in EXERCISES_DB.values() for ex in locs.values())

print("=" * 60)
print(" Ініціалізація баз даних FitCoach успішна!")
print("-" * 60)
print(f" Груп м'язів у базі: {len(EXERCISES_DB)} ({', '.join(EXERCISES_DB.keys())})")
print(f" Загальна кількість вправ: {total_exercises}")
print(f" Рівнів активності для TDEE: {len(ACTIVITY_LEVELS)}")
print("=" * 60)

 Ініціалізація баз даних FitCoach успішна!
------------------------------------------------------------
 Груп м'язів у базі: 6 (груди, спина, ноги, прес, руки, плечі)
 Загальна кількість вправ: 17
 Рівнів активності для TDEE: 5


##Інструменти (@tools)

In [ ]:
# ==========================================
# ІНСТРУМЕНТИ (TOOLS) — реалізовано 4 інструменти
# ==========================================
from langchain_core.tools import tool
import re

@tool
def bmi_calculator(query: str) -> str:
    """
    Розраховує індекс маси тіла (ІМТ/BMI) та надає інтерпретацію.

    Використовуй цей інструмент, коли користувач питає:
    - який його ІМТ;
    - чи є його вага в нормі;
    - про співвідношення ваги та зросту.

    Приклади запитів:
    - "розрахуй мій ІМТ, вага 75 кг, зріст 180 см"
    - "який мій ІМТ?" (якщо параметри вже згадувались у діалозі)
    """
    if not query.strip():
        return "Вкажи свою вагу (кг) та зріст (см) для розрахунку ІМТ."

    numbers = re.findall(r"\d+(?:[.,]\d+)?", query)
    if len(numbers) < 2:
        return (
            "Не вдалося визначити вагу та зріст. "
            "Вкажи, наприклад: 'вага 75 кг, зріст 180 см'."
        )

    weight_kg = float(numbers[0].replace(",", "."))
    height_cm = float(numbers[1].replace(",", "."))

    # Перевірка діапазонів (Guardrails)
    if weight_kg < 20 or weight_kg > 300:
        return f"Значення ваги {weight_kg} кг виглядає некоректним. Перевір введені дані."
    if height_cm < 100 or height_cm > 250:
        return f"Значення зросту {height_cm} см виглядає некоректним. Перевір введені дані."

    height_m = height_cm / 100
    bmi = weight_kg / (height_m ** 2)

    if bmi < 16:
        category, recommendation = "Виражений дефіцит маси тіла", "Рекомендується консультація лікаря та дієтолога."
    elif bmi < 18.5:
        category, recommendation = "Недостатня маса тіла", "Варто збільшити калорійність раціону, додати силові тренування."
    elif bmi < 25:
        category, recommendation = "Норма", "Підтримуй поточний режим харчування та активності."
    elif bmi < 30:
        category, recommendation = "Надмірна вага", "Рекомендується збільшити фізичну активність, переглянути раціон."
    else:
        category, recommendation = "Ожиріння", "Рекомендується консультація лікаря, поступове зниження ваги."

    return (
        f"Індекс маси тіла: {bmi:.1f}\n"
        f"Категорія: {category}\n"
        f"Рекомендація: {recommendation}\n\n"
        "Примітка: ІМТ не враховує співвідношення м'язової та жирової маси. "
        "Для спортсменів цей показник може бути неточним."
    )

@tool
def calorie_calculator(query: str) -> str:
    """
    Розраховує денну норму калорій (TDEE) та рекомендації для різних цілей.

    Використовуй цей інструмент, коли користувач хоче знати:
    - скільки калорій йому потрібно на день;
    - як харчуватися для схуднення або набору маси;
    - свій базовий метаболізм.

    Приклади запитів:
    - "розрахуй мою норму калорій: вага 75, зріст 180, 30 років, чоловік, помірна активність"
    - "скільки мені калорій на день для схуднення?"
    """
    if not query.strip():
        return (
            "Для розрахунку потрібні: вага (кг), зріст (см), вік (років), "
            "стать (чоловік/жінка), рівень активності."
        )

    query_lower = query.lower()
    numbers = re.findall(r"\d+(?:[.,]\d+)?", query)

    if len(numbers) < 3:
        return (
            "Не вдалося визначити всі параметри. "
            "Вкажи вагу, зріст і вік. Наприклад: 'вага 75 кг, зріст 180 см, 30 років'."
        )

    weight_kg = float(numbers[0].replace(",", "."))
    height_cm = float(numbers[1].replace(",", "."))
    age = int(float(numbers[2].replace(",", ".")))

    if any(w in query_lower for w in ["чоловік", "чол", " ч ", "male", " м "]):
        gender = "чоловік"
    elif any(w in query_lower for w in ["жінка", "жін", "female", " ж "]):
        gender = "жінка"
    else:
        gender = "чоловік"

    activity = "помірна"
    for level in ACTIVITY_LEVELS:
        if level in query_lower:
            activity = level
            break

    if weight_kg < 20 or weight_kg > 300:
        return f"Значення ваги {weight_kg} кг виглядає некоректним. Перевір введені дані."

    bmr = calculate_bmr(weight_kg, height_cm, age, gender)
    multiplier = ACTIVITY_LEVELS[activity]
    tdee = bmr * multiplier

    return (
        f"Базовий метаболізм (BMR): {bmr:.0f} ккал/день\n"
        f"Денна норма при '{activity}' активності (TDEE): {tdee:.0f} ккал/день\n\n"
        f"Рекомендації залежно від мети:\n"
        f"  Підтримка ваги: {tdee:.0f} ккал/день\n"
        f"  Схуднення (помірне): {tdee - 500:.0f} ккал/день (-0.5 кг/тиждень)\n"
        f"  Набір маси: {tdee + 300:.0f} ккал/день\n\n"
        f"Білки: {weight_kg * 1.6:.0f}–{weight_kg * 2.2:.0f} г/день\n"
        f"Жири: {tdee * 0.25 / 9:.0f}–{tdee * 0.35 / 9:.0f} г/день"
    )

@tool
def exercise_lookup(query: str) -> str:
    """
    Шукає вправи за групою м'язів та умовами (вдома або в залі).

    Використовуй цей інструмент, коли користувач:
    - питає про вправи на конкретну групу м'язів;
    - хоче тренуватися вдома або в залі;
    - просить підібрати вправи за рівнем складності.

    Приклади запитів:
    - "які вправи на груди вдома"
    - "підбери вправи на ноги для початківця"
    """
    available_groups = ", ".join(EXERCISES_DB.keys())

    if not query.strip():
        return f"Вкажи групу м'язів. Доступні: {available_groups}."

    query_lower = query.lower()

    muscle_group = None
    for group in EXERCISES_DB:
        if group in query_lower:
            muscle_group = group
            break

    if not muscle_group:
        return f"Групу м'язів не розпізнано. Доступні: {available_groups}."

    location = "зал" if "зал" in query_lower else "вдома"
    exercises = EXERCISES_DB[muscle_group].get(location, [])

    if not exercises:
        return f"Вправ для '{muscle_group}' ({location}) не знайдено в базі."

    result = [f"Вправи на {muscle_group} ({location}):"]
    for ex in exercises:
        result.append(f"\n{ex['назва']} ({ex['складність']})")
        result.append(f"  Техніка: {ex['техніка']}")
        result.append(f"  Підходи: {ex['підходи']} × {ex['повторення']}")

    return "\n".join(result)

@tool
def workout_timer(query: str) -> str:
    """
    Генерація таймера для інтервального тренування (HIIT/Табата).

    Використовуй цей інструмент, коли користувач:
    - просить скласти табату або HIIT;
    - хоче тренування на певний час (у хвилинах).

    Приклади запитів:
    - "склади мені табату на 15 хвилин"
    - "тренування HIIT на 20 хв"
    """
    numbers = re.findall(r"\d+", query)
    if not numbers:
        return "Вкажи бажану тривалість тренування у хвилинах."

    total_minutes = int(numbers[0])
    if total_minutes < 4 or total_minutes > 60:
        return "Рекомендований час тренування: від 4 до 60 хв."

    is_tabata = "табата" in query.lower() or "tabata" in query.lower()
    work_s, rest_s = (20, 10) if is_tabata else (40, 20)
    cycle_name = "Табата (20с робота / 10с відпочинок)" if is_tabata else "HIIT (40с робота / 20с відпочинок)"

    rounds = (total_minutes * 60) // (work_s + rest_s)

    return (
        f" План інтервального тренування:\n"
        f"Тип: {cycle_name}\n"
        f"Загальний час: {total_minutes} хв.\n"
        f"Кількість раундів: {rounds}\n"
        f"Цикл: працюйте {work_s}с, відпочивайте {rest_s}с."
    )

tools = [bmi_calculator, calorie_calculator, exercise_lookup, workout_timer]
print(f" {len(tools)} інструменти FitCoach створено.")

 4 інструменти FitCoach створено.


##Ініціалізація моделі

In [ ]:
# ==========================================
# ІНІЦІАЛІЗАЦІЯ LLM
# ==========================================

print(" Підключення до нейромережі OpenAI...")

chat_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
    max_tokens=512,
    timeout=30,
)

# Креативний тестовий запит для перевірки готовності
test_prompt = "Уяви, що ти енергійний фітнес-тренер. Одним коротким, мотиваційним реченням привітай мене на нашому першому тренуванні!"
test_response = chat_model.invoke(test_prompt)

print(" Модель успішно ініціалізовано та готова до роботи!")
print("-" * 60)
print(f" Перевірка зв'язку з FitCoach:\n «{test_response.content}»")
print("-" * 60)

 Підключення до нейромережі OpenAI...
 Модель успішно ініціалізовано та готова до роботи!
------------------------------------------------------------
 Перевірка зв'язку з FitCoach:
 «Привіт! Сьогодні ми розпочинаємо неймовірну подорож до твоєї найкращої версії – готовий до змін? 💪🔥»
------------------------------------------------------------


##Системний промпт і агент

In [ ]:
# ==========================================
# СТВОРЕННЯ АГЕНТА ТА ЕКСПРЕС-ТЕСТ
# ==========================================

SYSTEM_PROMPT = """
Ти - крутий та енергійний персональний фітнес-асистент FitCoach.

Твої головні завдання:
- Розраховувати ІМТ та денну норму калорій.
- Підбирати ефективні вправи для різних груп м'язів.
- Складати таймери для потужних інтервальних тренувань (HIIT/Табата).
- Відповідати українською мовою: чітко, структуровано та з мотивацією!

Твої залізні правила (Маршрутизація):
- Питання про ІМТ або вагу → використовуй [bmi_calculator]
- Питання про калорії, BMR, раціон → використовуй [calorie_calculator]
- Питання про вправи на групи м'язів → використовуй [exercise_lookup]
- Інтервальне тренування (HIIT/табата) → використовуй [workout_timer]

Обмеження (Guardrails):
- НІКОЛИ не давай медичних рекомендацій (ти тренер, а не лікар ).
- Якщо користувач назвав параметри (вага, зріст) - запам'ятай їх у контексті.
- Якщо запит не стосується фітнесу → ввічливо поверни розмову до спорту.
"""

print(" Збираємо та налаштовуємо агента FitCoach...")
agent = create_agent(
    model=chat_model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)
print(" Агент успішно створений та заряджений на роботу!\n")

# --- КРУТИЙ ЕКСПРЕС-ТЕСТ АГЕНТА ---
print("=" * 60)
print(" ЕКСПРЕС-ТЕСТ АГЕНТА (Перевірка виклику інструментів)")
print("=" * 60)

test_query = "Розрахуй мій ІМТ: я важу 85 кг, а зріст 182 см. Що скажеш?"
print(f" Користувач: {test_query}")
print(" FitCoach аналізує запит та викликає інструмент...\n")

test_msgs = [{"role": "user", "content": test_query}]
result = agent.invoke({"messages": test_msgs})
final_answer = result["messages"][-1].content

print(f" FitCoach:\n{final_answer}")
print("=" * 60)

 Збираємо та налаштовуємо агента FitCoach...
 Агент успішно створений та заряджений на роботу!

 ЕКСПРЕС-ТЕСТ АГЕНТА (Перевірка виклику інструментів)
 Користувач: Розрахуй мій ІМТ: я важу 85 кг, а зріст 182 см. Що скажеш?
 FitCoach аналізує запит та викликає інструмент...

 FitCoach:
Твій індекс маси тіла (ІМТ) становить 25.7, що відносить тебе до категорії "надмірна вага". Це означає, що варто звернути увагу на свою фізичну активність та раціон харчування.

Не забувай, що ІМТ не враховує співвідношення м'язової та жирової маси, тому для спортсменів цей показник може бути неточним. 

Якщо ти готовий до змін, я можу допомогти з підбором вправ або скласти план тренувань! 💪


 ## Інтерактивний чат (Жива сесія)

In [ ]:
# ==========================================
# ІНТЕРАКТИВНА СЕСІЯ (ЧАТ З FITCOACH)
# ==========================================

def run_fitcoach_session():
    print("\n" + "━" * 60)
    print("  БІОМЕТРИЧНИЙ ТЕРМІНАЛ FITCOACH АКТИВОВАНО ⚡ ")
    print("━" * 60)
    print(" Твій персональний AI-тренер готовий до роботи.")
    print(" Команди управління:")
    print("   [exit]   - завершити тренування")
    print("   [/reset] - скинути пам'ять (почати з чистого аркуша)")
    print("━" * 60)

    messages = []

    while True:
        try:
            user_input = input("\n👤 Ти: ").strip()
            if not user_input:
                continue

            if user_input.lower() in ("exit", "вихід", "/exit", "quit"):
                print("\n Тренування завершено! Гарного відновлення і до зустрічі!")
                break

            if user_input.lower() in ("/reset", "reset"):
                messages = []
                print("\n [СИСТЕМА]: Контекст скинуто. Параметри обнулено. Погнали знову!")
                continue

            messages.append({"role": "user", "content": user_input})

            # Динамічний статус завантаження (імітація для консолі)
            print(" FitCoach аналізує твої показники...", end="\r")

            result = agent.invoke({"messages": messages})
            messages = result["messages"]

            last = messages[-1]
            ans = last.content if hasattr(last, "content") else last.get("content", "")

            # Зачищаємо рядок завантаження і виводимо результат
            print(" " * 50, end="\r")
            print(f" FitCoach:\n{ans}\n")

        except KeyboardInterrupt:
            print("\n\n Екстрена зупинка. Сесію перервано. Побачимось!")
            break
        except Exception as e:
            print(f"\n [СИСТЕМНА ПОМИЛКА]: {e}")
            print("Спробуй переформулювати запит.")

# Для тестування діалогу в реальному часі просто розкоментуй рядок нижче:
run_fitcoach_session()


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  БІОМЕТРИЧНИЙ ТЕРМІНАЛ FITCOACH АКТИВОВАНО ⚡ 
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Твій персональний AI-тренер готовий до роботи.
 Команди управління:
   [exit]   - завершити тренування
   [/reset] - скинути пам'ять (почати з чистого аркуша)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

👤 Ти: Привіт
 FitCoach:
Привіт! Як я можу допомогти тобі сьогодні у твоєму фітнес-подорожі? 💪


👤 Ти: У мене надмірна вага чим зможеш допомогти
 FitCoach:
Я можу допомогти тобі з кількома аспектами:

1. **Розрахунок ІМТ** - визначимо, чи є твоя вага в нормі.
2. **Розрахунок калорій** - дізнаємося, скільки калорій тобі потрібно на день для схуднення.
3. **Вправи** - підберемо ефективні вправи для спалювання калорій та зміцнення м'язів.
4. **Інтервальні тренування** - складемо HIIT або Табату для інтенсивного тренування.

Якщо ти готовий, дай мені знати свою вагу та зріст, і ми почнемо!


👤 Ти: 98 кг і 

## Автоматичне тестування (10 сценаріїв)

In [ ]:
# ==========================================
# АВТОМАТИЧНЕ ТЕСТУВАННЯ
# ==========================================

def run_automatic_tests():
    test_queries = [
        "Привіт! Що ти вмієш?", # 1. Базовий запит
        "Розрахуй мій ІМТ: вага 80 кг, зріст 175 см", # 2. Виклик tool (ІМТ)
        "Розрахуй калорії: вага 80, зріст 175, 30 років, чоловік, помірна активність", # 3. Виклик tool (Калорії)
        "Які вправи на ноги в залі?", # 4. Виклик tool (Перевірка локації "зал")
        "Склади мені тренування табата на 15 хвилин", # 5. Виклик tool (Таймер)
        "Розрахуй норму калорій.", # 6. Некоректний запит (Обробка помилок)
        "Запам'ятай, що я важу 90 кг і мій зріст 180 см.", # 7. Запис у пам'ять
        "Який мій ІМТ?", # 8. Перевірка пам'яті (Використання контексту)
        "Які ліки пити при болю в спині?", # 9. Межовий сценарій (Медицина)
        "Як налаштувати Wi-Fi роутер?", # 10. Межовий сценарій (Поза темою)
    ]

    messages = []
    print("\n" + "=" * 60 + "\n АВТОМАТИЧНЕ ТЕСТУВАННЯ FitCoach\n" + "=" * 60)

    for i, query in enumerate(test_queries, start=1):
        print(f"\nТест #{i}\n" + "-" * 60 + f"\n👤 Запит: {query}")
        messages.append({"role": "user", "content": query})

        try:
            result_state = agent.invoke({"messages": messages})
            messages = result_state["messages"]
            last_message = messages[-1]
            answer = last_message.content if hasattr(last_message, "content") else last_message.get("content", "")
            print(" Відповідь FitCoach:\n", answer)
        except Exception as e:
            print(f" Помилка: {e}")

    # Фінальний вивід про успішне завершення!
    print("\n" + "=" * 60)
    print(" Усі 10 тестів пройдено! Тестування успішно завершено.")
    print("=" * 60)

run_automatic_tests()


 АВТОМАТИЧНЕ ТЕСТУВАННЯ FitCoach

Тест #1
------------------------------------------------------------
👤 Запит: Привіт! Що ти вмієш?
 Відповідь FitCoach:
 Привіт! Я твій персональний фітнес-асистент FitCoach. Я можу допомогти тобі з:

1. Розрахунком ІМТ (індексу маси тіла) та денній нормі калорій.
2. Підбором ефективних вправ для різних груп м'язів.
3. Складанням таймерів для потужних інтервальних тренувань (HIIT/Табата).

Якщо у тебе є питання про фітнес, не соромся запитувати! Давай досягнемо твоїх цілей разом! 💪

Тест #2
------------------------------------------------------------
👤 Запит: Розрахуй мій ІМТ: вага 80 кг, зріст 175 см
 Відповідь FitCoach:
 Твій індекс маси тіла (ІМТ) становить 26.1, що відносить тебе до категорії "Надмірна вага". 

Рекомендації:
- Збільшити фізичну активність.
- Переглянути свій раціон харчування.

Пам'ятай, що ІМТ не враховує співвідношення м'язової та жирової маси, тому для спортсменів цей показник може бути неточним. Якщо ти хочеш дізнатися більше 

## Таблиці

### Підсумкова продуктова таблиця

| Поле | Заповнення |
|---|---|
| **Назва бота** | FitCoach |
| **Для кого він створений** | Для новачків у фітнесі та людей, які тренуються самостійно (вдома чи в залі) і потребують швидкого планування без тренера. |
| **Яку задачу вирішує** | Швидкий персоналізований розрахунок норм харчування, підбір вправ та генерація таймерів через зручний діалог. |
| **Які tools реалізовано** | 1. `bmi_calculator` (ІМТ), 2. `calorie_calculator` (калорії), 3. `exercise_lookup` (вправи), 4. `workout_timer` (HIIT/Табата). |
| **Основна цінність для користувача** | Діалоговий stateful-інтерфейс: система "пам'ятає" параметри користувача протягом сесії, що позбавляє необхідності щоразу вводити свої дані заново. |
| **Головний ризик або обмеження** | Локальна статична база вправ не масштабується. Якщо запит написаний занадто плутано, Regex може не витягнути цифри. |
| **Що варто покращити перед реальним використанням** | Впровадити `Windowed memory` (видалення старих повідомлень з контексту) для економії токенів API. Підключити RAG (векторну БД) для динамічного пошуку вправ із відео. |

---

### Таблиця тестування

| № | Тип сценарію | Запит | Що перевіряється | Результат |
|---|---|---|---|---|
| 1 | Звичайне запитання | Привіт! Що ти вмієш? | Реакція без виклику tool | Описав свої функції (ІМТ, калорії, вправи, таймер). |
| 2 | Виклик інструмента | Розрахуй мій ІМТ: вага 80 кг, зріст 175 см | `bmi_calculator` | Успішно вивів ІМТ (26.1), категорію та попередження. |
| 3 | Виклик інструмента | Розрахуй калорії: вага 80, зріст 175, 30 років... | `calorie_calculator` | Розрахував BMR і TDEE для помірної активності. |
| 4 | Виклик інструмента | Які вправи на ноги в залі? | `exercise_lookup` (локація) | Знайшов групу "ноги", розпізнав "зал" і вивів жим та згинання в тренажері. |
| 5 | Виклик додаткового інструмента | Склади мені тренування табата на 15 хвилин | `workout_timer` | Згенерував план Табати (20с/10с) на 30 раундів. |
| 6 | Некоректний запит | Розрахуй норму калорій. | Обробка помилок у tool | Бот попросив вказати вагу, зріст і вік (не впав із помилкою). |
| 7 | Встановлення контексту | Запам'ятай, що я важу 90 кг і мій зріст 180 см. | Запис у `messages` | Підтвердив запам'ятовування параметрів. |
| 8 | Продовження контексту | Який мій ІМТ? | Використання пам'яті | Витягнув 90 кг і 180 см, успішно порахував ІМТ (27.8). |
| 9 | Межовий сценарій | Які ліки пити при болю в спині? | Медичні Guardrails | Відмовився давати медичні поради, скерував до лікаря. |
| 10 | Додаткова перевірка | Як налаштувати Wi-Fi роутер? | Дотримання ролі | Відмовився відповідати, наголосивши на фітнес-спеціалізації. |